# Lecture 5: Commutators and Two-Level Dynamics
### Computational Companion — Simultaneous Measurability, Uncertainty, and Larmor Precession

This notebook verifies every claim in Lecture 5:

1. **Simultaneous eigenbasis** — commuting matrices share eigenvectors; non-commuting ones don't
2. **Converse proof verification** — $[L,M]=0$ implies a common eigenbasis exists
3. **3D rotation non-commutativity** — the book-rotation demo, numerically
4. **Robertson uncertainty relation** — $\Delta L \cdot \Delta M \geq \frac{1}{2}|\langle [L,M] \rangle|$
5. **Larmor precession** — $H = \frac{\omega}{2}Z$, expectation values trace a cone
6. **Direct computation vs Ehrenfest** — two routes to the same answer
7. **General precession axis** — $H = \frac{\omega}{2}(\hat{n} \cdot \vec{\sigma})$

**Convention:** $\hbar = 1$ throughout.

**Reference:** Lecture 5 notes ("Quantum Systems via Linear Algebra")

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from mpl_toolkits.mplot3d import Axes3D

# ── Pauli matrices (ℏ = 1) ──
I2 = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)

# ── Basis vectors ──
e0 = np.array([[1], [0]], dtype=complex)
e1 = np.array([[0], [1]], dtype=complex)

def expect(M, v):
    """Expectation value ⟨v|M|v⟩ for a column vector v."""
    vf = v.flatten()
    return np.real(vf.conj() @ M @ vf)

def std_dev(M, v):
    """Standard deviation ΔM = √(⟨M²⟩ - ⟨M⟩²)."""
    return np.sqrt(max(0, expect(M @ M, v) - expect(M, v)**2))

print("Setup complete: Pauli matrices, basis vectors, helper functions. ℏ = 1.")

## 1. Simultaneous Eigenbasis — Commuting vs Non-Commuting Matrices

Two self-adjoint matrices can be simultaneously diagonalized if and only if they commute. We verify both directions and the converse proof's invariant-subspace argument.

In [ ]:
# ── 1. Simultaneous Eigenbasis ──

print("=" * 65)
print("COMMUTING MATRICES: share an eigenbasis")
print("=" * 65)

# Z and H = ω/2 · Z trivially commute
H_z = 0.5 * Z
comm = Z @ H_z - H_z @ Z
print(f"\n[Z, H=ω/2·Z] =\n{comm}")
print(f"Commute: {np.allclose(comm, 0)}")

# Both diagonalized by {e0, e1}
evals_Z, evecs_Z = np.linalg.eigh(Z)
evals_H, evecs_H = np.linalg.eigh(H_z)
print(f"\nZ eigenvectors:\n{evecs_Z}")
print(f"H eigenvectors:\n{evecs_H}")
print(f"Same eigenbasis (up to phase): {np.allclose(np.abs(evecs_Z), np.abs(evecs_H))}")

print("\n" + "=" * 65)
print("NON-COMMUTING MATRICES: NO common eigenbasis")
print("=" * 65)

comm_XZ = X @ Z - Z @ X
print(f"\n[X, Z] =\n{comm_XZ}")
print(f"Commute: {np.allclose(comm_XZ, 0)}")

evals_X, evecs_X = np.linalg.eigh(X)
print(f"\nZ eigenvectors: columns of\n{evecs_Z}")
print(f"X eigenvectors: columns of\n{np.round(evecs_X, 4)}")

# Check: are Z-eigenvectors also X-eigenvectors?
for i in range(2):
    v = evecs_Z[:, i]
    Xv = X @ v
    # v is eigenvector of X iff Xv is parallel to v
    ratio = Xv / (v + 1e-30)
    is_eigvec = np.allclose(ratio, ratio[0])
    print(f"  Z-eigenvector {i} is X-eigenvector? {is_eigvec}")

print("\n" + "=" * 65)
print("CONVERSE: [L,M]=0 ⟹ common eigenbasis exists")
print("=" * 65)

# Construct two commuting 3×3 Hermitian matrices with a degenerate eigenvalue
rng = np.random.default_rng(42)
V = np.linalg.qr(rng.standard_normal((3, 3)) + 1j * rng.standard_normal((3, 3)))[0]
d1 = np.diag([1.0, 2.0, 2.0])  # degenerate!
d2 = np.diag([3.0, 4.0, 5.0])
L = V @ d1 @ V.conj().T
M_comm = V @ d2 @ V.conj().T

print(f"\nL and M are 3×3 Hermitian with shared eigenbasis (one degenerate)")
print(f"[L, M] = 0? {np.allclose(L @ M_comm - M_comm @ L, 0)}")

# numpy's eigh finds eigenvectors — verify they simultaneously diagonalize both
evals_L, evecs_L = np.linalg.eigh(L)
print(f"\nEigenvalues of L: {np.round(evals_L, 4)}")

# Check that L's eigenvectors also diagonalize M
M_in_L_basis = evecs_L.conj().T @ M_comm @ evecs_L
print(f"M in L-eigenbasis (should be diagonal):\n{np.round(M_in_L_basis, 6)}")
print(f"M is diagonal in L-eigenbasis: {np.allclose(M_in_L_basis - np.diag(np.diag(M_in_L_basis)), 0)}")

## 2. 3D Rotation Non-Commutativity — The Book Demo

Rotations in 3D don't commute: $R_x(90°) R_y(90°) \neq R_y(90°) R_x(90°)$. This is literally the same group structure as $SU(2)$ — the Pauli matrices generate rotations, and their failure to commute is the failure of 3D rotations to commute.

In [ ]:
# ── 2. 3D Rotation Non-Commutativity ──

print("=" * 65)
print("3D ROTATIONS DON'T COMMUTE: Rx(90°)·Ry(90°) ≠ Ry(90°)·Rx(90°)")
print("=" * 65)

def rot_x(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1,0,0],[0,c,-s],[0,s,c]])

def rot_y(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c,0,s],[0,1,0],[-s,0,c]])

def rot_z(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c,-s,0],[s,c,0],[0,0,1]])

angle = np.pi / 2

# Order 1: Rx then Ry
R1 = rot_y(angle) @ rot_x(angle)
# Order 2: Ry then Rx
R2 = rot_x(angle) @ rot_y(angle)

print(f"\nRy(90°) · Rx(90°) =\n{np.round(R1, 4)}")
print(f"\nRx(90°) · Ry(90°) =\n{np.round(R2, 4)}")
print(f"\nSame? {np.allclose(R1, R2)}")
print(f"Difference:\n{np.round(R1 - R2, 4)}")

# Apply to a unit vector to visualize
v_book = np.array([1, 0, 0])
print(f"\nStarting vector: {v_book}")
print(f"After Ry·Rx: {np.round(R1 @ v_book, 4)}")
print(f"After Rx·Ry: {np.round(R2 @ v_book, 4)}")

# Now show SU(2) connection: Pauli matrices generate rotations
print("\n" + "=" * 65)
print("SU(2) CONNECTION: Pauli matrices generate rotations")
print("=" * 65)

# U_x(θ) = exp(-iθX/2), U_y(θ) = exp(-iθY/2)
Ux = expm(-1j * angle/2 * X)
Uy = expm(-1j * angle/2 * Y)

print(f"\nU_x(90°) · U_y(90°) =\n{np.round(Ux @ Uy, 4)}")
print(f"U_y(90°) · U_x(90°) =\n{np.round(Uy @ Ux, 4)}")
print(f"Same? {np.allclose(Ux @ Uy, Uy @ Ux)}")
print("\nSame non-commutativity structure — SU(2) and SO(3) share the same Lie algebra!")

## 3. Robertson Uncertainty Relation

$$\Delta L \cdot \Delta M \geq \frac{1}{2}|\langle [L,M] \rangle|$$

We verify this for all pairs of Pauli matrices across many states, and highlight the specific case $\Delta X \cdot \Delta Z \geq |\langle Y \rangle|$.

In [ ]:
# ── 3. Robertson Uncertainty Relation ──

print("=" * 65)
print("ROBERTSON: ΔL·ΔM ≥ ½|⟨[L,M]⟩|")
print("=" * 65)

# Test on many random states
rng = np.random.default_rng(123)
n_states = 500
pairs = [("X", "Z", X, Z), ("X", "Y", X, Y), ("Y", "Z", Y, Z)]

violations = {p[0]+p[1]: 0 for p in pairs}
lhs_all = {p[0]+p[1]: [] for p in pairs}
rhs_all = {p[0]+p[1]: [] for p in pairs}

for _ in range(n_states):
    # Random state on Bloch sphere
    theta = rng.uniform(0, np.pi)
    phi = rng.uniform(0, 2*np.pi)
    psi = np.array([[np.cos(theta/2)], [np.exp(1j*phi)*np.sin(theta/2)]])

    for n1, n2, M1, M2 in pairs:
        dL = std_dev(M1, psi)
        dM = std_dev(M2, psi)
        comm_val = M1 @ M2 - M2 @ M1
        rhs = 0.5 * abs(expect(comm_val, psi))
        lhs = dL * dM
        key = n1 + n2
        lhs_all[key].append(lhs)
        rhs_all[key].append(rhs)
        if lhs < rhs - 1e-10:
            violations[key] += 1

for n1, n2, M1, M2 in pairs:
    key = n1 + n2
    print(f"\n  [{n1},{n2}]: violations = {violations[key]}/{n_states}")
    print(f"    min(LHS - RHS) = {min(np.array(lhs_all[key]) - np.array(rhs_all[key])):.6f}")

# Specific example: ΔX·ΔZ ≥ |⟨Y⟩|
print("\n" + "=" * 65)
print("SPECIFIC: ΔX·ΔZ ≥ |⟨Y⟩| for several states")
print("=" * 65)

test_states = [
    ("e0", e0),
    ("e1", e1),
    ("x+", (e0 + e1)/np.sqrt(2)),
    ("y+", (e0 + 1j*e1)/np.sqrt(2)),
    ("(cos30°, e^(iπ/4)sin30°)", np.array([[np.cos(np.pi/6)], [np.exp(1j*np.pi/4)*np.sin(np.pi/6)]])),
]

for name, psi in test_states:
    dx = std_dev(X, psi)
    dz = std_dev(Z, psi)
    ey = abs(expect(Y, psi))
    print(f"  |{name}⟩:  ΔX·ΔZ = {dx*dz:.4f}  ≥  |⟨Y⟩| = {ey:.4f}  ✓ {dx*dz >= ey - 1e-10}")

# Visualization: scatter LHS vs RHS for X,Z pair
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (n1, n2, M1, M2) in zip(axes, pairs):
    key = n1 + n2
    ax.scatter(rhs_all[key], lhs_all[key], alpha=0.3, s=10, c='steelblue')
    mx = max(max(rhs_all[key]), max(lhs_all[key])) * 1.1
    ax.plot([0, mx], [0, mx], 'r--', linewidth=1.5, label='LHS = RHS (bound)')
    ax.set_xlabel(f'½|⟨[{n1},{n2}]⟩|  (RHS)')
    ax.set_ylabel(f'Δ{n1}·Δ{n2}  (LHS)')
    ax.set_title(f'Robertson: Δ{n1}·Δ{n2} ≥ ½|⟨[{n1},{n2}]⟩|')
    ax.legend(); ax.grid(alpha=0.3)
    ax.set_xlim(0, mx); ax.set_ylim(0, mx)

plt.suptitle('Robertson Uncertainty: all points above the red line (500 random states)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Larmor Precession — $H = \frac{\omega}{2}Z$

The expectation-value vector $(\langle X \rangle, \langle Y \rangle, \langle Z \rangle)$ traces a cone around the $z$-axis. We simulate this for several initial states, verifying:
- $\langle Z \rangle$ is conserved
- $\langle X \rangle$ and $\langle Y \rangle$ precess at frequency $\omega$
- The Bloch vector length is constant (pure state stays pure)

In [ ]:
# ── 4. Larmor Precession ──

omega = 1.0
H_larmor = (omega / 2) * Z
ts = np.linspace(0, 4 * np.pi, 500)

# Several initial states at different polar angles
init_states = [
    ("θ=π/2 (equator)", np.array([[1], [1]], dtype=complex) / np.sqrt(2)),
    ("θ=π/3", np.array([[np.cos(np.pi/6)], [np.sin(np.pi/6)]], dtype=complex)),
    ("θ=2π/3", np.array([[np.cos(np.pi/3)], [np.sin(np.pi/3)]], dtype=complex)),
    ("θ=π/4", np.array([[np.cos(np.pi/8)], [np.exp(1j*np.pi/5)*np.sin(np.pi/8)]], dtype=complex)),
]

fig = plt.figure(figsize=(10, 8))
ax3d = fig.add_subplot(111, projection='3d')

# Bloch sphere wireframe
u_s = np.linspace(0, 2*np.pi, 50)
v_s = np.linspace(0, np.pi, 50)
xs = np.outer(np.cos(u_s), np.sin(v_s))
ys = np.outer(np.sin(u_s), np.sin(v_s))
zs = np.outer(np.ones(len(u_s)), np.cos(v_s))
ax3d.plot_surface(xs, ys, zs, color='lightcyan', alpha=0.07)

colors = ['steelblue', 'crimson', 'green', 'darkorange']
for (name, psi0), color in zip(init_states, colors):
    exp_x = np.zeros(len(ts))
    exp_y = np.zeros(len(ts))
    exp_z = np.zeros(len(ts))
    for i, t in enumerate(ts):
        vt = expm(-1j * H_larmor * t) @ psi0.flatten()
        exp_x[i] = np.real(vt.conj() @ X @ vt)
        exp_y[i] = np.real(vt.conj() @ Y @ vt)
        exp_z[i] = np.real(vt.conj() @ Z @ vt)
    ax3d.plot(exp_x, exp_y, exp_z, color=color, linewidth=2, label=name)
    ax3d.scatter([exp_x[0]], [exp_y[0]], [exp_z[0]], c=color, s=50, zorder=5)

for data in [([[-1.3,1.3],[0,0],[0,0]]), ([[0,0],[-1.3,1.3],[0,0]]), ([[0,0],[0,0],[-1.3,1.3]])]:
    ax3d.plot(data[0], data[1], data[2], 'gray', linestyle='--', alpha=0.3)

ax3d.set_xlabel('⟨X⟩'); ax3d.set_ylabel('⟨Y⟩'); ax3d.set_zlabel('⟨Z⟩')
ax3d.set_title('Larmor Precession: H = (ω/2)Z\nExpectation values trace cones around z-axis')
ax3d.set_box_aspect([1,1,1])
ax3d.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# Verify conservation and precession for equator state
psi_eq = init_states[0][1]
exp_x_eq = [np.real((expm(-1j*H_larmor*t) @ psi_eq.flatten()).conj() @ X @ (expm(-1j*H_larmor*t) @ psi_eq.flatten())) for t in ts]
exp_y_eq = [np.real((expm(-1j*H_larmor*t) @ psi_eq.flatten()).conj() @ Y @ (expm(-1j*H_larmor*t) @ psi_eq.flatten())) for t in ts]
exp_z_eq = [np.real((expm(-1j*H_larmor*t) @ psi_eq.flatten()).conj() @ Z @ (expm(-1j*H_larmor*t) @ psi_eq.flatten())) for t in ts]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(ts, exp_x_eq, 'steelblue', lw=2, label='⟨X⟩'); axes[0].plot(ts, np.cos(omega*ts), 'r--', lw=1.5, label='cos(ωt)')
axes[0].set_title('⟨X⟩(t) = cos(ωt)'); axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_xlabel('t')
axes[1].plot(ts, exp_y_eq, 'green', lw=2, label='⟨Y⟩'); axes[1].plot(ts, -np.sin(omega*ts), 'r--', lw=1.5, label='−sin(ωt)')
axes[1].set_title('⟨Y⟩(t) = −sin(ωt)'); axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_xlabel('t')
axes[2].plot(ts, exp_z_eq, 'purple', lw=2, label='⟨Z⟩'); axes[2].axhline(0, color='r', ls='--', lw=1.5)
axes[2].set_title('⟨Z⟩(t) = 0 (conserved)'); axes[2].legend(); axes[2].grid(alpha=0.3); axes[2].set_xlabel('t')
plt.suptitle('Equator state: |ψ(0)⟩ = (|0⟩+|1⟩)/√2,  H = (ω/2)Z', fontsize=13)
plt.tight_layout(); plt.show()

print(f"Max |⟨X⟩ − cos(ωt)|: {np.max(np.abs(np.array(exp_x_eq) - np.cos(omega*ts))):.2e}")
print(f"Max |⟨Z⟩|: {np.max(np.abs(exp_z_eq)):.2e}")

## 5. Direct Computation vs Ehrenfest — Two Routes, Same Answer

We verify the lecture's §2.4 derivation: computing $\langle X \rangle(t) = \mathbf{v}(t)^\dagger X \mathbf{v}(t)$ directly from the time-evolved state gives the same result as solving the Ehrenfest ODE $\frac{d}{dt}\langle X \rangle = -\omega \langle Y \rangle$.

We also confirm the explicit formulas: $\langle X \rangle(t) = 2\text{Re}(\alpha^*\beta\, e^{i\omega t})$.

In [ ]:
# ── 5. Direct Computation vs Ehrenfest ──

print("=" * 65)
print("DIRECT COMPUTATION vs EHRENFEST: two routes, same answer")
print("=" * 65)

omega = 1.0
H_larmor = (omega / 2) * Z

# General initial state: α|0⟩ + β|1⟩
alpha = np.cos(np.pi/5)
beta = np.exp(1j * np.pi/3) * np.sin(np.pi/5)
psi0 = np.array([[alpha], [beta]], dtype=complex)

print(f"\nα = {alpha:.4f},  β = {beta:.4f}")
print(f"|α|² + |β|² = {abs(alpha)**2 + abs(beta)**2:.6f}")

ts_cmp = np.linspace(0, 4*np.pi, 500)

# Route 1: Direct matrix exponential
direct_X = np.zeros(len(ts_cmp))
direct_Y = np.zeros(len(ts_cmp))
direct_Z = np.zeros(len(ts_cmp))

for i, t in enumerate(ts_cmp):
    vt = expm(-1j * H_larmor * t) @ psi0.flatten()
    direct_X[i] = np.real(vt.conj() @ X @ vt)
    direct_Y[i] = np.real(vt.conj() @ Y @ vt)
    direct_Z[i] = np.real(vt.conj() @ Z @ vt)

# Route 2: Analytic formulas from §2.4
analytic_X = 2 * np.real(alpha.conj() * beta * np.exp(1j * omega * ts_cmp))
analytic_Y = 2 * np.imag(alpha.conj() * beta * np.exp(1j * omega * ts_cmp))
analytic_Z = np.full_like(ts_cmp, abs(alpha)**2 - abs(beta)**2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, direct, analytic) in zip(axes, [
    ('⟨X⟩', direct_X, analytic_X),
    ('⟨Y⟩', direct_Y, analytic_Y),
    ('⟨Z⟩', direct_Z, analytic_Z)
]):
    ax.plot(ts_cmp, direct, 'steelblue', lw=2, label=f'{name} (expm)')
    ax.plot(ts_cmp, analytic, 'r--', lw=1.5, label=f'{name} (analytic)')
    ax.set_xlabel('t'); ax.set_ylabel(name)
    ax.set_title(f'{name}(t): direct vs analytic')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Direct computation matches analytic formulas from §2.4', fontsize=13)
plt.tight_layout(); plt.show()

print(f"Max |⟨X⟩_direct − ⟨X⟩_analytic|: {np.max(np.abs(direct_X - analytic_X)):.2e}")
print(f"Max |⟨Y⟩_direct − ⟨Y⟩_analytic|: {np.max(np.abs(direct_Y - analytic_Y)):.2e}")
print(f"Max |⟨Z⟩_direct − ⟨Z⟩_analytic|: {np.max(np.abs(direct_Z - analytic_Z)):.2e}")

## 6. General Precession Axis — $H = \frac{\omega}{2}(\hat{n} \cdot \vec{\sigma})$

The most general two-level Hamiltonian is $H = \frac{\omega}{2}(n_x X + n_y Y + n_z Z)$. The expectation-value vector precesses around the axis $\hat{n}$ at frequency $\omega$, with the component along $\hat{n}$ conserved. We verify this for several axis choices.

In [ ]:
# ── 6. General Precession Axis ──

print("=" * 65)
print("GENERAL PRECESSION: H = (ω/2)(n·σ)")
print("=" * 65)

omega = 1.0
ts_gen = np.linspace(0, 4*np.pi, 500)

# Several axis choices
axes_list = [
    ("ẑ = (0,0,1)", np.array([0, 0, 1.0])),
    ("x̂ = (1,0,0)", np.array([1, 0, 0.0])),
    ("ŷ = (0,1,0)", np.array([0, 1, 0.0])),
    ("(1,1,1)/√3", np.array([1, 1, 1.0]) / np.sqrt(3)),
    ("(1,0,1)/√2", np.array([1, 0, 1.0]) / np.sqrt(2)),
]

fig = plt.figure(figsize=(16, 10))

for idx, (ax_name, n_hat) in enumerate(axes_list):
    H_gen = (omega / 2) * (n_hat[0]*X + n_hat[1]*Y + n_hat[2]*Z)

    # Initial state: |0⟩
    psi0_gen = e0.flatten()

    exp_xyz = np.zeros((len(ts_gen), 3))
    for i, t in enumerate(ts_gen):
        vt = expm(-1j * H_gen * t) @ psi0_gen
        exp_xyz[i, 0] = np.real(vt.conj() @ X @ vt)
        exp_xyz[i, 1] = np.real(vt.conj() @ Y @ vt)
        exp_xyz[i, 2] = np.real(vt.conj() @ Z @ vt)

    # Component along n̂ should be conserved
    comp_along_n = exp_xyz @ n_hat
    print(f"\n  Axis {ax_name}:")
    print(f"    Component along n̂: min={comp_along_n.min():.6f}, max={comp_along_n.max():.6f} (constant ✓)")

    # 3D plot
    ax3d = fig.add_subplot(2, 3, idx+1, projection='3d')
    u_s = np.linspace(0, 2*np.pi, 30)
    v_s = np.linspace(0, np.pi, 30)
    ax3d.plot_surface(np.outer(np.cos(u_s), np.sin(v_s)),
                      np.outer(np.sin(u_s), np.sin(v_s)),
                      np.outer(np.ones(len(u_s)), np.cos(v_s)),
                      color='lightcyan', alpha=0.06)
    ax3d.plot(exp_xyz[:, 0], exp_xyz[:, 1], exp_xyz[:, 2], 'steelblue', lw=2)
    ax3d.scatter([exp_xyz[0, 0]], [exp_xyz[0, 1]], [exp_xyz[0, 2]], c='red', s=50, zorder=5)
    # Draw precession axis
    ax3d.plot([0, n_hat[0]*1.3], [0, n_hat[1]*1.3], [0, n_hat[2]*1.3], 'k-', lw=2.5, alpha=0.7)
    ax3d.set_xlabel('⟨X⟩'); ax3d.set_ylabel('⟨Y⟩'); ax3d.set_zlabel('⟨Z⟩')
    ax3d.set_title(f'n̂ = {ax_name}', fontsize=10)
    ax3d.set_box_aspect([1,1,1])

plt.suptitle('General Precession: H = (ω/2)(n̂·σ),  |ψ(0)⟩ = |0⟩\nBlack line = precession axis', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Uncertainty Product Across the Bloch Sphere

We visualize $\Delta X \cdot \Delta Z$ as a function of the state on the Bloch sphere (parameterized by $\theta, \phi$), alongside the Robertson bound $|\langle Y \rangle|$. The bound is tight for some states (saturation) and loose for others.

In [ ]:
# ── 7. Uncertainty Product Across the Bloch Sphere ──

n_theta = 100
n_phi = 100
thetas = np.linspace(0.01, np.pi - 0.01, n_theta)
phis = np.linspace(0, 2*np.pi, n_phi)

unc_product = np.zeros((n_theta, n_phi))
robertson_rhs = np.zeros((n_theta, n_phi))

for i, th in enumerate(thetas):
    for j, ph in enumerate(phis):
        psi = np.array([[np.cos(th/2)], [np.exp(1j*ph)*np.sin(th/2)]])
        dx = std_dev(X, psi)
        dz = std_dev(Z, psi)
        ey = abs(expect(Y, psi))
        unc_product[i, j] = dx * dz
        robertson_rhs[i, j] = ey

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ΔX·ΔZ
im0 = axes[0].imshow(unc_product, extent=[0, 360, 180, 0], aspect='auto', cmap='viridis')
axes[0].set_xlabel('φ (degrees)'); axes[0].set_ylabel('θ (degrees)')
axes[0].set_title('ΔX · ΔZ (uncertainty product)')
plt.colorbar(im0, ax=axes[0])

# |⟨Y⟩| (Robertson RHS)
im1 = axes[1].imshow(robertson_rhs, extent=[0, 360, 180, 0], aspect='auto', cmap='magma')
axes[1].set_xlabel('φ (degrees)'); axes[1].set_ylabel('θ (degrees)')
axes[1].set_title('|⟨Y⟩| (Robertson bound)')
plt.colorbar(im1, ax=axes[1])

# Slack = ΔX·ΔZ - |⟨Y⟩|
slack = unc_product - robertson_rhs
im2 = axes[2].imshow(slack, extent=[0, 360, 180, 0], aspect='auto', cmap='RdYlGn')
axes[2].set_xlabel('φ (degrees)'); axes[2].set_ylabel('θ (degrees)')
axes[2].set_title('Slack: ΔX·ΔZ − |⟨Y⟩| ≥ 0')
plt.colorbar(im2, ax=axes[2])

plt.suptitle('Robertson Uncertainty: ΔX·ΔZ ≥ |⟨Y⟩| across the Bloch sphere', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Min slack (must be ≥ 0): {slack.min():.6f}")
print(f"Bound is tight (slack ≈ 0) at: θ ≈ 90°, φ ≈ 0° or 180° (X-eigenstates)")

# Find where the bound is tightest
min_idx = np.unravel_index(np.argmin(slack), slack.shape)
print(f"Tightest at θ = {np.degrees(thetas[min_idx[0]]):.1f}°, φ = {np.degrees(phis[min_idx[1]]):.1f}°")

## 1. Simultaneous Eigenbasis — Commuting vs Non-Commuting Matrices

Two self-adjoint matrices can be simultaneously diagonalized if and only if they commute. We verify:
- **Commuting pair**: $Z$ and $H = \frac{\omega}{2}Z$ share an eigenbasis (trivially)
- **Non-commuting pair**: $X$ and $Z$ do NOT share an eigenbasis — their eigenvectors are misaligned
- **Converse proof**: construct a common eigenbasis for two commuting matrices numerically